<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [3]:
pip install -r requirements.txt

  Using cached ipykernel-6.29.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached ipympl-0.9.8-py3-none-any.whl.metadata (8.9 kB)
  Using cached numpy-2.4.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-2.2.3.tar.gz (4.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [12 lines of output]
      + meson setup C:\Users\User\AppData\Local\Temp\pip-install-r9ilfqtt\pandas_28920a9a79d54cf0bb39a9b69d14d21b C:\Users\User\AppData\Local\Temp\pip-install-r9ilfqtt\pandas_28920a9a79d54cf0bb39a9b69d14d21b\.mesonpy-uus2t3el\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\User\AppData\Local\Temp\pip-install-r9ilfqtt\pandas_28920a9a79d54cf0bb39a9b69d14d21b\.mesonpy-uus2t3el\build\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.1
      Source dir: C:\Users\User\AppData\Local\Temp\pip-install-r9ilfqtt\pandas_28920a9a79d54cf0bb39a9b69d14d21b
      Build dir: C:\Users\User\AppData\Local\Temp\pip-install-r9ilfqtt\pandas_28920a9a79d54cf0bb39a9b69d14d21b\.mesonpy-uus2t3el\build
      Build type: native build
      Project name: pandas
      Project version: 2.2.3
      
  

In [7]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np

# DataFrame structure used to store and analyze results across multiple runs and configurations.
import pandas as pd

# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt

# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random

# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2

# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod

# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy

# OS is used for file path manipulations and directory creation.
import os

# Time is used to measure the runtime of the algorithm and to timestamp output files.
import time

In [8]:
class Solution(ABC):
    """Generic abstract class for an optimization solution."""

    def __init__(self, repr=None):
        # If no representation is provided, create a random one using the subclass method.
        if repr is None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

In [9]:
class VermeerSolution(Solution):
    """A candidate painting made from 100 colored triangles."""

    def __init__(self, target_image, repr=None):
        # Store the original image that the generated triangle image will be compared against.
        self.target_image = target_image

        # Project canvas size: 300x400 pixels.
        self.width = 300
        self.height = 400

        # Project requirement: use 100 triangles.
        self.num_triangles = 100

        # Calls Solution.__init__, which either stores a provided repr or creates a random one.
        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates a random genome of 100 small triangles.
        Each row has: [x1, y1, x2, y2, x3, y3, r, g, b]
        """

        # Matrix with one row per triangle and 9 values per triangle.
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)

        for i in range(self.num_triangles):
            # Pick a random center points in the canvas for the triangle
            cx = randint(0, self.width)
            cy = randint(0, self.height)

            # Define an max radius for the triangles
            radius = 40
            # Generate 3 random x-coordinates and 3 random y-coordinates within the range of the defined radius
            # This allows to control the size of the randomly intialized traingles (can start with small triangles)
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            # Generate one random RGB color for the whole triangle.
            r, g, b = [randint(0, 255) for _ in range(3)]
            # Store the triangle's vertices and color in the genome.
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]

        return random_repr

    def render_canvas(self):
        """Converts the 100-triangle genome into an actual image matrix."""

        # Start from a black image with shape: height x width x RGB channels.
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        for gene in self.repr:
            # Extract the three triangle vertices in OpenCV's expected format.
            pts = np.array(
                [[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32
            )

            # Reshape points so cv2.fillPoly can draw the triangle.
            pts = pts.reshape((-1, 1, 2))

            # Extract the triangle color.
            # NOTE: The loaded OpenCV image is BGR, but the project logic treats these as channel values consistently.
            color = (int(gene[6]), int(gene[7]), int(gene[8]))

            # Draw a filled triangle on the canvas.
            cv2.fillPoly(canvas, [pts], color)

        return canvas

    def fitness(self):
        """Calculates RMSE between the target image and the generated triangle image.
        Applies a small penalty to overlapping triangles in order to try and resolve the problem with facial features
        """

        # Render this individual's genome into an image.
        generated_image = self.render_canvas()

        # Convert to float before subtracting, avoiding uint8 overflow/underflow errors.
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # Pixel-by-pixel difference between target and generated image.
        diff = target_float - gen_float
        # RMSE = sqrt(mean(square(error))). Lower RMSE means a better image.
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        # The final fitness is the visual error PLUS the geometric penalty
        return rmse

In [12]:
# Path to João's local copy of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
rafa_path = r"..\Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
gui_path_2 = r"C:\Users\User\Documents\GitHub\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(gui_path_2)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {gui_path_2}")

# Create one random triangle painting to test the VermeerSolution class.
first_painting = VermeerSolution(target_image=original_image)

# Print its genome and RMSE fitness to confirm the representation and fitness function work.
print(f"{first_painting.repr} has fitness {first_painting.fitness()}")

[[253 121 209 128 254  70  25 206  79]
 [  7 322  49 311  20 324 188 119 219]
 [223 176 239 118 195 142   4 150 253]
 [175 265 229 241 234 212 197 144 252]
 [111 149  78 155  83 168  54  97 117]
 [199 191 180 171 226 176 101 102 240]
 [220 391 280 393 250 346 246 233 210]
 [162 357 166 344 158 314  94 253 168]
 [177 158 168 235 184 194 194 130 193]
 [198 115 177 129 194  75  36 251 226]
 [155 324 141 320  90 348 246 248  78]
 [ 35 215  62 238   9 231 231  75  45]
 [183  71 167 136 178  75  56 114  17]
 [300 128 300 141 300 139 129 156  86]
 [289  69 250  58 226  20 120  48   8]
 [ 29 316  26 300  36 303 181  21  78]
 [ 53 317  37 368  10 307  10 203 197]
 [254 167 272 155 219 134 209  68  44]
 [ 44  54  39  83 101 118 237 146  87]
 [ 76 282  39 277  87 254  95  71 251]
 [300 267 285 250 291 250 138 109 100]
 [228  66 296  33 273  66 177  70 224]
 [196 340 243 343 182 365  94 225 182]
 [  3 237  52 227   0 184 248  21 127]
 [ 82 362  64 389  71 367 110 209 104]
 [127  64 134  63 155 117

In [5]:
# Path to local copies of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
rafa_path = r"Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
# Load the original image using OpenCV.
original_image = cv2.imread(gui_path)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {gui_path}")

# Create one random triangle painting to test the VermeerSolution class.
first_painting = VermeerSolution(target_image=original_image)

# Print its genome and RMSE fitness to confirm the representation and fitness function work.
print(f"{first_painting.repr} has fitness {first_painting.fitness()}")

[[  0 237   0 196  35 232 251 164 113]
 [  4 368  60 400  60 400 192 103 118]
 [ 68 256  65 281 126 274 218  41  73]
 [ 50 274  25 331  89 266 107 135 250]
 [ 25 223  52 239  35 294 138  71  29]
 [260 142 246 137 251 144 232 193  91]
 [149  42 143   9 156  11  68 110 221]
 [246 133 252 141 238 117 172 155 159]
 [157  18 158  20 156  17 106 198 160]
 [ 65 336  61 329  84 329 118 202  91]
 [ 57 284  16 322  14 272 211  30 193]
 [233  47 252  46 274  74  60 103  56]
 [ 21 159  11  99  23 129 167  56  13]
 [  0 254  21 260   0 238 235  43  37]
 [215 194 199 231 235 191  98  32 198]
 [ 68 148 117 219 121 177  17  26 249]
 [ 33   0  64   1  96  29  42 120 227]
 [291  23 300  15 299   0  69 135 239]
 [107  98 119  39 134  37 241  58 224]
 [173   0 146  26 152  44  43 109 154]
 [113 189 156 173 101 232  98 155  55]
 [ 39 182  35 171   5 163 122 214  99]
 [155 396 155 400 138 391 159  80  85]
 [236 142 228 124 276  87 175  38 117]
 [104 387 123 363 104 329 115 146 162]
 [  0 191   0 154   0 148

Now to test out if the initial class functions in producing an image and comparing it to the original painting

In [6]:
"""gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'

original_vermeer = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')

my_first_painting = VermeerSolution(target_image=original_vermeer)

score = round(my_first_painting.fitness(), 2)

print(f'The RMSE Score of random painting is: {score}')"""

"gonçalo_path = 'C:\\CIfO\\Project\\data\\girl_pearl_earing.png'\n\noriginal_vermeer = cv2.imread(r'C:\\CIfO\\Project\\data\\girl_pearl_earing.png')\n\nmy_first_painting = VermeerSolution(target_image=original_vermeer)\n\nscore = round(my_first_painting.fitness(), 2)\n\nprint(f'The RMSE Score of random painting is: {score}')"

#### Vemeer 3

In [13]:
class VermeerGA3:
    """
    GA variant with:
    - Ranking + Tournament Selection
    - Cycle or OX (Two-point) crossover or Uniform crossover
    - Nudge/Replace Mutation with added Scramble + inversion mutation
    - Manhattan distance for fitness sharing (instead of Euclidean)
    """

    def __init__(
        self,
        target_image,
        pop_size=150,
        generations=2000,
        pc=0.85,
        pm=0.03,
        elitism_size=10,
        tournament_size=4,
        selection_method="ranking",  # "tournament" or "ranking"
        crossover_method="cycle",  # "cycle" or "ox" or "uniform"
        mutation_method="swap",  # "swap" or "scramble" or "inversion"
        use_crossover=True,
        use_mutation=True,
        use_fitness_sharing=False,
        variance_threshold=450,
        save_every=None,
        output_dir=None,
    ):

        self.target_image = target_image
        self.pop_size = pop_size
        self.generations = generations
        self.pc = pc
        self.pm = pm
        self.elitism_size = elitism_size

        # Genetic Operators
        self.selection_method = selection_method
        self.tournament_size = tournament_size
        self.crossover_method = crossover_method
        self.mutation_method = mutation_method

        self.use_crossover = use_crossover
        self.use_mutation = use_mutation
        self.use_fitness_sharing = use_fitness_sharing
        self.variance_threshold = variance_threshold
        self.save_every = save_every
        self.output_dir = output_dir

        # Initial random population
        self.population = [
            VermeerSolution(target_image=self.target_image)
            for _ in range(self.pop_size)
        ]

    # ---------- Fitness sharing with Manhattan distance ----------

    def apply_fitness_sharing(self):
        """
        Fitness sharing using Manhattan (L1) distance between individuals.
        Penalizes crowded solutions to avoid premature convergence.
        """
        radius_share = 0.15
        alpha = 1.0

        flat_reprs = np.vstack([ind.repr.flatten() for ind in self.population])

        # Manhattan distance between all pairs
        diff = flat_reprs[:, np.newaxis, :] - flat_reprs[np.newaxis, :, :]
        dist_matrix = np.sum(np.abs(diff), axis=-1)

        # Normalize distances to [0, 1]
        min_dist = np.min(dist_matrix)
        max_dist = np.max(dist_matrix)
        if max_dist == min_dist:
            norm_dist_matrix = np.zeros_like(dist_matrix)
        else:
            norm_dist_matrix = (dist_matrix - min_dist) / (max_dist - min_dist)

        # Similarity from distance
        similarity_matrix = np.where(
            norm_dist_matrix < radius_share,
            1.0 - (norm_dist_matrix / radius_share) ** alpha,
            0.0,
        )

        niche_counts = np.sum(similarity_matrix, axis=1)

        for idx, ind in enumerate(self.population):
            ind.shared_fitness = ind.fitness_score * niche_counts[idx]

    # ---------- Evaluation ----------

    def evaluate_population(self):
        """Evaluate and sort population by fitness (best first)."""
        for ind in self.population:
            if not hasattr(ind, "fitness_score"):
                ind.fitness_score = ind.fitness()

        if self.use_fitness_sharing:
            self.apply_fitness_sharing()
            self.population.sort(key=lambda ind: ind.shared_fitness)
        else:
            self.population.sort(key=lambda ind: ind.fitness_score)

    # ---------- Selection Methods ----------
    def select_parent(self):
        """Dispatcher between tournament and ranking selection."""
        if self.selection_method == "ranking":
            return self.ranking_selection()
        return self.tournament_selection()

    # ---------- Tournament selection ----------

    def tournament_selection(self):
        """Picks K random individuals and returns the one with the lowest RMSE."""

        # Randomly select tournament_size candidates.
        tournament = sample(self.population, self.tournament_size)

        if self.use_fitness_sharing:
            winner = sorted(tournament, key=lambda ind: ind.shared_fitness)[0]
        else:
            # The candidate with lowest RMSE wins the tournament.
            winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    # ---------- Ranking selection ----------

    def ranking_selection(self):
        """
        Linear ranking selection.
        Population is sorted ascending (best at index 0), so the best
        individual gets the highest rank weight = n, worst gets 1.
        """
        n = len(self.population)
        ranks = np.arange(n, 0, -1)  # [n, n-1, ..., 1]
        probs = ranks / ranks.sum()
        idx = np.random.choice(n, p=probs)
        return self.population[idx]

    # ---------- Crossover Methods ----------

    def crossover(self, parent1, parent2):
        """Dispatcher for crossover methods."""
        if self.crossover_method == "two_point":
            return self.two_point_crossover(parent1, parent2)
        elif self.crossover_method == "cycle":
            return self.cycle_crossover(parent1, parent2)
        # Default to Uniform
        return self.uniform_crossover(parent1, parent2)

    def uniform_crossover(self, parent1, parent2):
        """Uniform crossover: each triangle is inherited from either Parent 1 or Parent 2."""
        # Create a child object (random_representation of a solution(child)); then replace its genome using parent genes.
        child1 = VermeerSolution(
            target_image=self.target_image,
        )
        child2 = VermeerSolution(
            target_image=self.target_image,
        )
        # Mask has one 0/1 value per triangle.
        # 1 means take triangle from parent1; 0 means take triangle from parent2.
        mask = np.random.randint(0, 2, size=(child1.num_triangles, 1))
        # np.where applies the mask triangle by triangle.
        child1.repr = np.where(mask, parent1.repr, parent2.repr).copy()
        child2.repr = np.where(mask, parent2.repr, parent1.repr).copy()
        
        return child1, child2

    def cycle_crossover(self, parent1, parent2):
        child1 = VermeerSolution(target_image=self.target_image)
        child2 = VermeerSolution(target_image=self.target_image)
        n = child1.num_triangles

        positions = list(range(n))
        np.random.shuffle(positions)

        new_repr1 = np.empty_like(parent1.repr)
        new_repr2 = np.empty_like(parent1.repr)
        
        i = 0
        use_p1 = True
        while i < n:
            cycle_len = randint(1, max(1, n // 4))
            block = positions[i : i + cycle_len]
            
            source1 = parent1 if use_p1 else parent2
            source2 = parent2 if use_p1 else parent1
            
            for pos in block:
                new_repr1[pos] = source1.repr[pos]
                new_repr2[pos] = source2.repr[pos]
                
            use_p1 = not use_p1
            i += cycle_len

        child1.repr = new_repr1.copy()
        child2.repr = new_repr2.copy()
        
        return child1, child2

    def two_point_crossover(self, parent1, parent2):
        """
        Order crossover (OX) (two-point crossover), adapted.
        Standard OX copies a segment from parent1 then fills the rest from
        parent2 in order, skipping duplicates. Here triangles are unique
        to each parent so nothing is ever skipped; OX becomes a clean
        two-cut segment crossover, which preserves its spirit.
        """
        child1 = VermeerSolution(target_image=self.target_image)
        child2 = VermeerSolution(target_image=self.target_image)
        n = child1.num_triangles

        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))

        child1.repr = parent2.repr.copy()
        child1.repr[c1:c2] = parent1.repr[c1:c2]

        child2.repr = parent1.repr.copy()
        child2.repr[c1:c2] = parent2.repr[c1:c2]
        
        return child1, child2

    # ---------- Mutation ----------

    def mutate(self, child):
        """
        With probability pm, apply either scramble or inversion mutation
        on the triangle order. Note: pm here is per individual (segment op),
        not per triangle as in GA2.
        """
        # 1. Standard Nudge/Replace Mutation (Crucial for exploring shapes/colors!)
        for i in range(child.num_triangles):
            if random() < self.pm:
                mutation_type = random()
                if mutation_type < 0.90:
                    # 90% of mutations are small nudges to preserve useful structures.
                    child.repr[i][0:6] += np.random.randint(-10, 11, 6)  # coordinates
                    child.repr[i][6:9] += np.random.randint(-15, 16, 3)  # color

                    # Keep x-coordinates inside the image width.
                    child.repr[i][0] = np.clip(child.repr[i][0], 0, child.width)
                    child.repr[i][2] = np.clip(child.repr[i][2], 0, child.width)
                    child.repr[i][4] = np.clip(child.repr[i][4], 0, child.width)

                    # Keep y-coordinates inside the image height.
                    child.repr[i][1] = np.clip(child.repr[i][1], 0, child.height)
                    child.repr[i][3] = np.clip(child.repr[i][3], 0, child.height)
                    child.repr[i][5] = np.clip(child.repr[i][5], 0, child.height)

                    # Keep color channels valid.
                    child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)
                    pass
                else:
                    # Full triangle replace
                    child.repr[i] = [
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, child.width),
                        randint(0, child.height),
                        randint(0, 255),
                        randint(0, 255),
                        randint(0, 255),
                    ]
                    pass
        if random() < self.pm:
            if self.mutation_method == "scramble":
                child = self.scramble_mutation(child)
            elif self.mutation_method == "inversion":
                child = self.inversion_mutation(child)

        if hasattr(child, "fitness_score"):
            del child.fitness_score
        return child

    def scramble_mutation(self, child):
        """Shuffle the triangle order inside a random sub-segment."""
        n = child.num_triangles
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))
        segment = child.repr[c1:c2].copy()
        np.random.shuffle(segment)
        child.repr[c1:c2] = segment
        return child

    def inversion_mutation(self, child):
        """Reverse the triangle order inside a random sub-segment."""
        n = child.num_triangles
        c1, c2 = sorted(np.random.choice(n, size=2, replace=False))
        child.repr[c1:c2] = child.repr[c1:c2][::-1]
        return child

    # ---------- Diversity tracking ----------

    def calculate_population_variance(self):
        """Mean genotypic variance across all genes in the population."""
        flat_reprs = np.vstack([ind.repr.flatten() for ind in self.population])
        gene_variances = np.var(flat_reprs, axis=0)
        return np.mean(gene_variances)

    def maybe_save_generation(self, gen):
        """Optionally save the current best image every N generations."""
        if self.save_every is None or self.output_dir is None:
            return
        if gen % self.save_every != 0:
            return

        import os

        os.makedirs(self.output_dir, exist_ok=True)
        image = self.population[0].render_canvas()
        cv2.imwrite(f"{self.output_dir}/gen_{gen}.png", image)

    # ---------- Main loop ----------

    def run(self):
        """Main evolutionary loop."""
        print(f"Starting Evolution for {self.generations} generations...")

        fitness_history = []
        variance_history = []

        self.evaluate_population()

        for gen in range(self.generations):
            best_current_solution = min(
                self.population, key=lambda ind: ind.fitness_score
            )
            fitness_history.append(best_current_solution.fitness_score)

            if gen % 50 == 0 or gen == self.generations - 1:
                current_variance = self.calculate_population_variance()
                
            variance_history.append(current_variance)

            # Dynamic fitness sharing
            if current_variance < self.variance_threshold:
                if not self.use_fitness_sharing:
                    print(
                        f"\n Variance dropped to {current_variance:.2f} "
                        f"Fitness Sharing Activated at Gen: {gen}"
                    )
                self.use_fitness_sharing = True
                self.apply_fitness_sharing()

            if gen % 50 == 0:
                print(
                    f"Generation {gen} | Best RMSE: "
                    f"{best_current_solution.fitness_score:.2f} | " 
                    f"Variance: {current_variance:.2f}"
                )
                self.maybe_save_generation(gen)

            # Elitism
            elites = sorted(self.population, key=lambda ind: ind.fitness_score)[:self.elitism_size]

            next_generation = [
                deepcopy(ind) for ind in elites
            ]

            # Fill the rest with offspring
            while len(next_generation) < self.pop_size:
                p1 = self.select_parent()

                if self.use_crossover and random() < self.pc:
                    p2 = self.select_parent()
                    child1, child2 = self.crossover(p1, p2)

                    if self.use_mutation:
                        child1 = self.mutate(child1)
                        child2 = self.mutate(child2)

                    next_generation.append(child1)
                    if len(next_generation) < self.pop_size:
                        next_generation.append(child2)
                else:
                    child = deepcopy(p1)
                    if self.use_mutation:
                        child = self.mutate(child)
                    next_generation.append(child)

            self.population = next_generation
            self.evaluate_population()

        self.evaluate_population()
        
        best_final_solution = min(self.population, key=lambda ind: ind.fitness_score)
        print(
            f"Evolution Complete! Final RMSE: "
            f"{best_final_solution.fitness_score:.2f}"
        )
        return best_final_solution, fitness_history, variance_history

In [7]:
target = cv2.imread(r"C:\CIfO\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png")
pop_size = 100
generations = 500
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
save_every = 500
variance_threshold = 550
use_fitness_sharing = True

In [8]:
# 1. Define Base Directory
base_results_dir = r"C:\CIfO\Project\data\results"

# Create base directory if it does not exist
os.makedirs(base_results_dir, exist_ok=True)

# 2. Create New Folder to be used for each run
existing_runs = [
    folder
    for folder in os.listdir(base_results_dir)
    # Folders will be organized by the Nº of Run, so first folder will be the first trial Run then subsequent runs will be 2, 3, 4 ...
    if folder.startswith("Run")
]
# Create list to track the number of runs performed
run_numbers = []

for folder in existing_runs:
    try:
        number = int(folder.replace("Run", ""))
        run_numbers.append(number)
    except:
        pass
    # Looks at the last run performed and adds the next run max(run_numbers, default=0) + 1
next_run_number = max(run_numbers, default=0) + 1

run_folder = os.path.join(base_results_dir, f"Run{next_run_number}")

os.makedirs(run_folder, exist_ok=True)

print(f"Results will be saved in: {run_folder}")

Results will be saved in: C:\CIfO\Project\data\results\Run2


#### Test run of the final class VermeerGA3

In [9]:
ga = VermeerGA3(
    target_image=target,
    pop_size=pop_size,
    generations=generations,
    pc=pc,
    pm=pm,
    elitism_size=elitism_size,
    crossover_method="cycle",  # or "ox"
    use_fitness_sharing=use_fitness_sharing,
    variance_threshold=variance_threshold,
    save_every=save_every,
    output_dir=run_folder,
)

best_painting, fitness_history, variance_history = ga.run()

Starting Evolution for 500 generations...
Generation 0 | Best RMSE: 99.07 | Variance: 8947.42
Generation 50 | Best RMSE: 62.58 | Variance: 2198.40


KeyboardInterrupt: 

In [ ]:
# Baseline Experimment Implementation
exp_name = "Control_Baseline"
notes = "Standard benchmark run (Tournament + Uniform + Scramble). Fitness Sharing turned off for natural variation testing."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "uniform"
mutation = "scramble"
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # --- NOVO: Preparar o dicionário de histórico pedido ---
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # --- NOVO: Gravação detalhada de Hiperparâmetros ---
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(fitness_history, color="black", linewidth=1.5)
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished. Best final RMSE: {np.mean(final_rmses):.2f}"
)

min_len = min(len(r) for r in all_runs_history)

# 2. Cortar todas as runs pelo mesmo tamanho para evitar erros no Numpy
aligned_history = [r[:min_len] for r in all_runs_history]
history_matrix = np.array(aligned_history)

# 3. Guardar dados raw com o tamanho dinâmico (min_len) em vez de (generations + 1)
raw_df = pd.DataFrame(history_matrix)   
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(min_len)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

# 4. Guardar resumo estatístico também com o tamanho dinâmico
stats_df = pd.DataFrame(
    {
        "Generation": np.arange(min_len),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[23:38:57] Starting batch: Control_Baseline (1 Runs)
Run 1/1...
Starting Evolution for 20 generations...
Generation 0 | Best RMSE: 100.06 | Variance: 7964.50
Evolution Complete! Final RMSE: 100.06

[23:38:57] Batch Control_Baseline finished. Best final RMSE: 100.06


In [ ]:
# Baseline Experimment Implementation
exp_name = "Ranking Selection"
notes = "Change made was selection operator from tournament to ranking selection. All other parameters remain the same as the control baseline for direct comparison."

# Algorithm Hyperparameters
selection = "ranking"
crossover = "uniform"
mutation = "scramble"
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # --- NOVO: Preparar o dicionário de histórico pedido ---
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # --- NOVO: Gravação detalhada de Hiperparâmetros ---
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(fitness_history, color="black", linewidth=1.5)
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished. Best final RMSE: {np.mean(final_rmses):.2f}"
)

min_len = min(len(r) for r in all_runs_history)

# 2. Cortar todas as runs pelo mesmo tamanho para evitar erros no Numpy
aligned_history = [r[:min_len] for r in all_runs_history]
history_matrix = np.array(aligned_history)

# 3. Guardar dados raw com o tamanho dinâmico (min_len) em vez de (generations + 1)
raw_df = pd.DataFrame(history_matrix)   
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(min_len)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

# 4. Guardar resumo estatístico também com o tamanho dinâmico
stats_df = pd.DataFrame(
    {
        "Generation": np.arange(min_len),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)

In [ ]:
# Genetic Algorithm Crossover Implementation
exp_name = "Crossover_Variant_TwoPoint"
notes = "Crossover Insulation: Testing with Two-Point (OX) to evaluate layer construction. Tournament Selection and Swap Mutation."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "two_point"  # <-- Change Made
mutation = "scramble"
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(
        fitness_history, color="green", linewidth=1.5
    )  # Cor Verde para o Two-Point
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished. Mean final RMSE: {np.mean(final_rmses):.2f}"
)

min_len = min(len(r) for r in all_runs_history)

# 2. Cortar todas as runs pelo mesmo tamanho para evitar erros no Numpy
aligned_history = [r[:min_len] for r in all_runs_history]
history_matrix = np.array(aligned_history)

# 3. Guardar dados raw com o tamanho dinâmico (min_len) em vez de (generations + 1)
raw_df = pd.DataFrame(history_matrix)   
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(min_len)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

# 4. Guardar resumo estatístico também com o tamanho dinâmico
stats_df = pd.DataFrame(
    {
        "Generation": np.arange(min_len),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[00:02:00] Starting batch: Crossover_Variant_TwoPoint (20 Runs)
Run 1/20...
Starting Evolution for 2000 generations...
Generation 0 | Best RMSE: 99.91 | Variance: 8955.75
Generation 50 | Best RMSE: 55.13 | Variance: 779.00

 Variance dropped to 385.99 Fitness Sharing Activated at Gen: 100
Generation 100 | Best RMSE: 49.14 | Variance: 385.99
Generation 150 | Best RMSE: 48.03 | Variance: 3334.78
Generation 200 | Best RMSE: 46.22 | Variance: 3548.36
Generation 250 | Best RMSE: 44.72 | Variance: 3281.37
Generation 300 | Best RMSE: 43.70 | Variance: 3271.42
Generation 350 | Best RMSE: 43.00 | Variance: 3586.80
Generation 400 | Best RMSE: 42.48 | Variance: 3305.91
Generation 450 | Best RMSE: 41.62 | Variance: 3705.88
Generation 500 | Best RMSE: 41.28 | Variance: 4071.73
Generation 550 | Best RMSE: 40.64 | Variance: 3937.56
Generation 600 | Best RMSE: 40.05 | Variance: 3912.99
Generation 650 | Best RMSE: 39.74 | Variance: 3791.55
Generation 700 | Best RMSE: 39.74 | Variance: 3919.74
Generati

In [ ]:
# Implementation Configuration for Mutation Variant: Inversion
exp_name = "Mutation_Variant_Inversion"
notes = "Mutation Isolation: Inversion test to evaluate drastic space exploration jumps. Tournament Selection and Crossover Uniform."

# Algorithm Hyperparameters
selection = "tournament"
crossover = "uniform"
mutation = "inversion"  # <-- Change Made
n_runs = 20

pop_size = 150
generations = 2000
pc = 0.85
pm = 0.03
elitism_size = 10
tournament_size = 4
variance_threshold = 550
use_fitness_sharing = False

print(f"\n[{time.strftime('%H:%M:%S')}] Starting batch: {exp_name} ({n_runs} Runs)")

variant_dir = os.path.join("data", "results", exp_name)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

for run in range(1, n_runs + 1):
    print(f"Run {run}/{n_runs}...")

    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)

    ga = VermeerGA3(
        target_image=original_image,
        pop_size=pop_size,
        generations=generations,
        pc=pc,
        pm=pm,
        elitism_size=elitism_size,
        selection_method=selection,
        crossover_method=crossover,
        mutation_method=mutation,
        use_fitness_sharing=use_fitness_sharing,
        tournament_size=tournament_size, 
        variance_threshold=variance_threshold 
    )

    best_sol, fitness_history, variance_history = ga.run()

    # Preparar o dicionário de histórico
    history_dict = {}
    for gen in range(len(fitness_history)):
        history_dict[gen] = {
            "fitness": fitness_history[gen],
            "variance": (
                variance_history[gen]
                if gen < len(variance_history)
                else variance_history[-1]
            ),
        }

    # Gravação detalhada de Hiperparâmetros
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"GA3 Hyperparameters for {exp_name} | Run Number: {run}\n\n")

        if notes:
            file.write(f"Notes for this run: {notes}\n\n")

        file.write(f"--- VARIANT SETUP ---\n")
        file.write(f"Selection Method: {selection}\n")
        file.write(f"Crossover Method: {crossover}\n")
        file.write(f"Mutation Method: {mutation}\n\n")

        file.write(f"--- BASE PARAMETERS ---\n")
        file.write(f"Population Size: {pop_size}\n")
        file.write(f"Generations: {generations}\n")
        file.write(f"Crossover Probability (pc): {pc}\n")
        file.write(f"Mutation Probability (pm): {pm}\n")
        file.write(f"Elitism Size: {elitism_size}\n")
        file.write(f"Tournament Size: {tournament_size}\n")
        file.write(f"Fitness Sharing: {use_fitness_sharing}\n")
        file.write(f"Variance Threshold for Fitness Sharing: {variance_threshold}\n\n")

        file.write(f"--- GENERATION HISTORY ---\n")
        for gen, metrics in history_dict.items():
            if gen % 50 == 0:
                fit = metrics["fitness"]
                var = metrics["variance"]
                file.write(f"Gen {gen} | Fitness: {fit:.2f} | Variance: {var:.2f}\n")

        file.write(f"\n--- FINAL RESULTS ---\n")
        file.write(f"Final RMSE Score: {best_sol.fitness_score:.2f}\n")
        file.write(f"Final Variance: {variance_history[-1]:.2f}\n")

    # Guardar a imagem e o gráfico
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sol.render_canvas())

    plt.figure(figsize=(8, 4))
    plt.plot(
        fitness_history, color="red", linewidth=1.5
    )  # Cor Vermelha para o Scramble
    plt.title(f"Convergence - {exp_name} Run {run} (Final: {fitness_history[-1]:.2f})")
    plt.xlabel("Generation")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches="tight")
    plt.close()

    all_runs_history.append(fitness_history)
    final_rmses.append(fitness_history[-1])

print(
    f"\n[{time.strftime('%H:%M:%S')}] Batch {exp_name} finished. Best final RMSE: {np.mean(final_rmses):.2f}"
)

min_len = min(len(r) for r in all_runs_history)

# 2. Cortar todas as runs pelo mesmo tamanho para evitar erros no Numpy
aligned_history = [r[:min_len] for r in all_runs_history]
history_matrix = np.array(aligned_history)

# 3. Guardar dados raw com o tamanho dinâmico (min_len) em vez de (generations + 1)
raw_df = pd.DataFrame(history_matrix)   
raw_df.index = [f"Run_{i+1}" for i in range(n_runs)]
raw_df.columns = [f"Gen_{i}" for i in range(min_len)]
raw_df.to_csv(os.path.join(variant_dir, "raw_fitness_data.csv"))

# 4. Guardar resumo estatístico também com o tamanho dinâmico
stats_df = pd.DataFrame(
    {
        "Generation": np.arange(min_len),
        "Mean_RMSE": np.mean(history_matrix, axis=0),
        "Std_Dev": np.std(history_matrix, axis=0),
        "Best_RMSE": np.min(history_matrix, axis=0),
    }
)
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[15:22:27] Starting batch: Mutation_Variant_Inversion (30 Runs)
Run 1/30...
Starting Evolution for 2000 generations...
Generation 0 | Best RMSE: 98.73 | Variance: 8929.14
Generation 50 | Best RMSE: 54.62 | Variance: 978.02
Generation 100 | Best RMSE: 47.55 | Variance: 842.56

 Variance dropped to 534.22 Fitness Sharing Activated at Gen: 150
Generation 150 | Best RMSE: 43.79 | Variance: 534.22
Generation 200 | Best RMSE: 42.89 | Variance: 2200.99
Generation 250 | Best RMSE: 41.72 | Variance: 2152.41
Generation 300 | Best RMSE: 40.29 | Variance: 2176.12
Generation 350 | Best RMSE: 39.46 | Variance: 2349.13
Generation 400 | Best RMSE: 38.96 | Variance: 2658.21
Generation 450 | Best RMSE: 37.53 | Variance: 2643.70
Generation 500 | Best RMSE: 36.83 | Variance: 2015.35
Generation 550 | Best RMSE: 36.12 | Variance: 2275.54
Generation 600 | Best RMSE: 35.14 | Variance: 2550.22
Generation 650 | Best RMSE: 34.18 | Variance: 2524.03
Generation 700 | Best RMSE: 34.32 | Variance: 2471.39
Generatio

KeyboardInterrupt: 

In [24]:
import os
import numpy as np
import pandas as pd

def recover_statistical_data(exp_name="Selection_Variant_Ranking", num_runs=30):
    print(f"A iniciar recuperação de dados para: {exp_name}...")
    variant_dir = os.path.join("data", "results", exp_name)
    
    recovered_history = []
    
    for run in range(1, num_runs + 1):
        filepath = os.path.join(variant_dir, f"Run_{run}", "hyperparameters.txt")
        run_rmse = []
        
        if not os.path.exists(filepath):
            print(f"⚠️ Ficheiro não encontrado para a Run {run}!")
            continue
            
        with open(filepath, 'r') as f:
            for line in f:
                # Procura as linhas com o histórico normal
                if line.startswith("Gen "):
                    try:
                        parts = line.split("|")
                        fitness_part = parts[1].strip()
                        rmse_val = float(fitness_part.split(":")[1].strip())
                        run_rmse.append(rmse_val)
                    except Exception as e:
                        pass
                
                # NOVO: Procura a linha com o resultado final e adiciona-a como Gen 2000
                elif line.startswith("Final RMSE Score:"):
                    try:
                        # Pega na linha "Final RMSE Score: 30.41", divide nos ":" e guarda o "30.41"
                        final_rmse = float(line.split(":")[1].strip())
                        run_rmse.append(final_rmse)
                    except Exception as e:
                        pass
        
        recovered_history.append(run_rmse)
        print(f"Run {run} extraída. ({len(run_rmse)} pontos de dados)")

    # Converter para numpy array
    history_matrix = np.array(recovered_history)
    
    # Calcular as estatísticas
    mean_rmse = np.mean(history_matrix, axis=0)
    std_rmse = np.std(history_matrix, axis=0)
    best_rmse = np.min(history_matrix, axis=0)
    
    actual_length = history_matrix.shape[1]
    gens = np.arange(0, actual_length * 50, 50)  # Gera os números de geração correspondentes

    # Guardar no CSV!
    stats_df = pd.DataFrame({
        "Generation": gens,
        "Mean_RMSE": mean_rmse,
        "Std_Dev": std_rmse,
        "Best_RMSE": best_rmse
    })
    
    csv_path = os.path.join(variant_dir, "statistical_summary_ranking_selection.csv")
    stats_df.to_csv(csv_path, index=False)
    
    print(f"\n✅ Recuperação de Desastre Concluída!")
    print(f"CSV guardado em: {csv_path}")
    print(f"Melhor Média Final (Gen 2000): {mean_rmse[-1]:.2f}")

# Executa a recuperação
recover_statistical_data()

A iniciar recuperação de dados para: Selection_Variant_Ranking...
Run 1 extraída. (41 pontos de dados)
Run 2 extraída. (41 pontos de dados)
Run 3 extraída. (41 pontos de dados)
Run 4 extraída. (41 pontos de dados)
Run 5 extraída. (41 pontos de dados)
Run 6 extraída. (41 pontos de dados)
Run 7 extraída. (41 pontos de dados)
Run 8 extraída. (41 pontos de dados)
Run 9 extraída. (41 pontos de dados)
Run 10 extraída. (41 pontos de dados)
Run 11 extraída. (41 pontos de dados)
Run 12 extraída. (41 pontos de dados)
Run 13 extraída. (41 pontos de dados)
Run 14 extraída. (41 pontos de dados)
Run 15 extraída. (41 pontos de dados)
Run 16 extraída. (41 pontos de dados)
Run 17 extraída. (41 pontos de dados)
Run 18 extraída. (41 pontos de dados)
Run 19 extraída. (41 pontos de dados)
Run 20 extraída. (41 pontos de dados)
Run 21 extraída. (41 pontos de dados)
Run 22 extraída. (41 pontos de dados)
Run 23 extraída. (41 pontos de dados)
Run 24 extraída. (41 pontos de dados)
Run 25 extraída. (41 pontos de 

In [25]:
df = pd.read_csv(r"C:\CIfO\CIFO_Group37_Vestigial\Project\data\results\Selection_Variant_Ranking\statistical_summary_ranking_selection.csv")
df.tail()

,Generation,Mean_RMSE,Std_Dev,Best_RMSE
36,1800,32.297000,1.381369,29.09
37,1850,32.205000,1.438183,29.05
38,1900,32.133333,1.477129,28.91
39,1950,32.051333,1.482888,28.96
40,2000,31.926667,1.483470,28.81
